In [75]:
import openai 
from qdrant_client import QdrantClient
import os

In [76]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [77]:
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_5160/3214170885.py:1: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(


In [78]:
def retrieve_data(query, qdrant_client=qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="amazon_reviews_collection",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieve_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    retrieve_context_details = []
    retrieve_context_product_names = []
    retrieve_context_helpful_votes = []

    for result in results.points:
        payload = result.payload or {}
        retrieved_context_ids.append(payload.get('product_id'))
        retrieve_context.append(payload.get('review_text', ''))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(payload.get('rating', 0))
        retrieve_context_helpful_votes.append(payload.get('helpful_votes', 0))
        retrieve_context_product_names.append(payload.get('product_name', ''))
        retrieve_context_details.append(payload.get('details', ''))

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieve_context': retrieve_context,
        'similarity_scores': similarity_scores,
        'retrieved_context_ratings': retrieved_context_ratings,
        'retrieve_context_details': retrieve_context_details,
        'retrieve_context_product_names': retrieve_context_product_names,
        'retrieve_context_helpful_votes': retrieve_context_helpful_votes
    }

In [79]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False


In [80]:
retrieve_context = retrieve_data('What kind of Yoga Mat do you offer?', k=10)

In [81]:
retrieve_context

{'retrieved_context_ids': ['B006',
  'B002',
  'B002',
  'B009',
  'B007',
  'B003',
  'B003',
  'B001',
  'B007',
  'B001'],
 'retrieve_context': ['This product has feels premium. solid choice.',
  'This product has feels premium. solid choice.',
  'This product has feels premium. great product.',
  'This product has customer support was helpful. amazing quality.',
  'This product has customer support was helpful. amazing quality.',
  'This product has feels premium. not bad at all.',
  'This product has feels premium. pretty decent.',
  'This product has feels premium. pretty decent.',
  'This product has feels premium. satisfied with purchase.',
  'This product has feels premium. worth buying.'],
 'similarity_scores': [0.26275313,
  0.26275313,
  0.24808408,
  0.22430849,
  0.22430375,
  0.22286105,
  0.22154877,
  0.22154376,
  0.21658917,
  0.2107374],
 'retrieved_context_ratings': [4, 4, 1, 4, 2, 4, 3, 5, 3, 3],
 'retrieve_context_details': [{'Date First Available': 'November 11,

Format retrieved context function

In [93]:
def process_context(context):
    formatted_context = []
    for id, chunk, rating, details, product_name, helpful_votes in zip(context['retrieved_context_ids'], context['retrieve_context'], context['retrieved_context_ratings'], context['retrieve_context_details'], context['retrieve_context_product_names'], context['retrieve_context_helpful_votes']):
        formatted_context.append(f"ID: {id}\nRating: {rating}\nReview: {chunk}\nDetails: {details}\nProduct Name: {product_name}\nHelpful Votes: {helpful_votes}\n---")
    return "\n".join(formatted_context)

In [94]:
preprocessed_context = process_context(retrieve_context)

In [95]:
print(preprocessed_context)

ID: B006
Rating: 4
Review: This product has feels premium. solid choice.
Details: {'Date First Available': 'November 11, 2024', 'Manufacturer': 'Brand B', 'ASIN': 'ASIN7115185'}
Product Name: Yoga Mat
Helpful Votes: 27
---
ID: B002
Rating: 4
Review: This product has feels premium. solid choice.
Details: {'Date First Available': 'July 11, 2024', 'Manufacturer': 'Brand A', 'ASIN': 'ASIN1551295'}
Product Name: Stainless Steel Water Bottle
Helpful Votes: 45
---
ID: B002
Rating: 1
Review: This product has feels premium. great product.
Details: {'Date First Available': 'January 14, 2024', 'Manufacturer': 'Brand A', 'ASIN': 'ASIN2743629'}
Product Name: Stainless Steel Water Bottle
Helpful Votes: 5
---
ID: B009
Rating: 4
Review: This product has customer support was helpful. amazing quality.
Details: {'Date First Available': 'March 25, 2024', 'Manufacturer': 'Brand D', 'ASIN': 'ASIN9467237'}
Product Name: Backpack
Helpful Votes: 46
---
ID: B007
Rating: 2
Review: This product has customer suppo

Create Prompt Function

In [96]:
def build_prompt(preprocessed_context, question):
    prompt = f"""You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      {preprocessed_context}
      Question: {question}
      Answer:"""
    return prompt

In [97]:
prompt = build_prompt(preprocessed_context, "What kind of BackPack do you offer?")
print(prompt)

You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      ID: B006
Rating: 4
Review: This product has feels premium. solid choice.
Details: {'Date First Available': 'November 11, 2024', 'Manufacturer': 'Brand B', 'ASIN': 'ASIN7115185'}
Product Name: Yoga Mat
Helpful Votes: 27
---
ID: B002
Rating: 4
Review: This product has feels premium. solid choice.
Details: {'Date First Available': 'July 11, 2024', 'Manufacturer': 'Brand A', 'ASIN': 'ASIN1551295'}
Product Name: Stainless Steel Water Bottle
Helpful Votes: 45
---
ID: B002
Rating: 1
Review: This product has feels premium. great product.
Details: {'Date First Available': 'January 14,

Generate Answer Function

In [98]:
def gen_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [99]:
print(gen_answer(prompt))

We offer a Backpack from Brand D (ASIN ASIN9467237). It was first available on March 25, 2024, has a 4-star rating, and 46 helpful votes.


Combined RAG Pipeline

In [100]:
def Rag_Pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))
    retrieve_context = retrieve_data(question, qdrant_client=qdrant_client, k=top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = gen_answer(prompt)
    return answer

In [101]:
Rag_Pipeline("What kind of Backpack do you offer which has helpful votes greater than 20?", top_k=5)

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_5160/1312403169.py:2: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))


'The Backpack from Brand B (ASIN ASIN5116319) has 41 helpful votes.'